# Baltimore LVT by SDAT Assessment Group — Three Per-Zone 4:1 Split-Rate Models

Maryland SDAT reassesses on a triennial cycle, dividing Baltimore City into three assessment groups (`ASSESGRP` = 1, 2, 3), each revalued every third year. This notebook treats **each assessment group as its own taxing district**: three independent revenue-neutral 4:1 split-rate solves, each exported and reported as its own "city" (`baltimore_group1`, `baltimore_group2`, `baltimore_group3`).

Because each group's levy stays within the group, no burden transfers across groups — which also neutralizes the assessment-lag inequity between groups that were last revalued in different years.

Policy assumptions (identical to the citywide model except for the per-group revenue neutrality):

- Scope: **Baltimore city levy only**
- Reform: **4:1 split-rate**, revenue-neutral **within each assessment group separately**
- Existing exemptions preserved: fully exempt where there is no current city bill (`ARTAXBAS <= 0` or `CITY_TAX <= 0`); these parcels are **excluded** from the modeled universe, exports, and graphics
- Current tax = observed `CITY_TAX` (gross city levy before parcel-level credits), per Baltimore's derived-millage pattern
- Land/improvement split: `ARTAXBAS` allocated by current full cash shares (`CURRLAND`/`CURRIMPR`) — portal-verified basis (2026-06-11); the `BFCV*` base-cycle fields are NULL-land for ~46% of parcels and are not used

Data notes:

- Parcels: Baltimore City `Realproperty_OB` ArcGIS layer, reusing the citywide model's cache in `../baltimore/data/` (March 2026 vintage)
- `ASSESGRP` was not in the cached field list, so it is pulled separately from the live layer (June 2026) and joined by `PIN`. Assessment-group geography is static, so the vintage mix is safe; every PIN maps to exactly one group in the live data.

In [ ]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from shapely.geometry import Polygon

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

CITY_BASE = 'baltimore_group'
STATE_FIPS = '24'
COUNTY_FIPS = '510'
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0
GROUPS = ['1', '2', '3']

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Step 2: Load citywide parcel data

Same cache-reuse pattern as `cities/baltimore_transit/model.ipynb`.

In [ ]:
BALTIMORE_DATA = Path('../baltimore/data')
PARCEL_QUERY_URL = 'https://geodata.baltimorecity.gov/egis/rest/services/CityView/Realproperty_OB/FeatureServer/0/query'
ATTR_FIELDS = [
    'OBJECTID', 'PIN', 'CURRLAND', 'CURRIMPR', 'EXMPLAND', 'EXMPIMPR',
    'BFCVLAND', 'BFCVIMPR', 'ARTAXBAS', 'CITY_TAX', 'CITYCRED', 'CCREDAMT',
    'STATCRED', 'SCREDAMT', 'ZONECODE', 'USEGROUP', 'DHCDUSE1', 'DWELUNIT',
    'NO_IMPRV', 'VACIND', 'OWNER_1', 'OWNER_2', 'FULLADDR', 'NEIGHBOR',
    'PROPDESC', 'YEAR_BUILD', 'STRUCTAREA', 'LOT_SIZE',
]


def _latest(pattern):
    hits = sorted(list(DATA_DIR.glob(pattern)) + list(BALTIMORE_DATA.glob(pattern)), key=lambda p: p.name)
    return hits[-1] if hits else None


def fetch_arcgis_records(query_url, out_fields=None, chunk_size=1000, return_geometry=False):
    session = requests.Session()
    count = session.get(
        query_url, params={'f': 'json', 'where': '1=1', 'returnCountOnly': 'true'}, timeout=60
    ).json()['count']
    print(f'Total records: {count:,}')
    rows = []
    for offset in range(0, count, chunk_size):
        params = {
            'f': 'json', 'where': '1=1',
            'outFields': '*' if out_fields is None else ','.join(out_fields),
            'returnGeometry': str(return_geometry).lower(),
            'resultOffset': offset, 'resultRecordCount': chunk_size,
            'orderByFields': 'OBJECTID ASC',
        }
        if return_geometry:
            params.update({'outSR': 4326, 'geometryPrecision': 6})
        features = session.get(query_url, params=params, timeout=180).json().get('features', [])
        if not features:
            break
        rows.extend(features)
    return rows


attrs_cache = _latest('baltimore_attrs_*.parquet')
if attrs_cache is not None:
    parcel_attrs = pd.read_parquet(attrs_cache)
    print(f'Loaded attribute cache: {attrs_cache}')
else:
    feats = fetch_arcgis_records(PARCEL_QUERY_URL, out_fields=ATTR_FIELDS)
    parcel_attrs = (
        pd.DataFrame([f['attributes'] for f in feats])
        .drop_duplicates('OBJECTID').sort_values('OBJECTID').reset_index(drop=True)
    )
    parcel_attrs.to_parquet(DATA_DIR / 'baltimore_attrs_fresh.parquet', index=False)

geometry_cache = _latest('baltimore_geometry_*.parquet')
if geometry_cache is not None:
    parcel_geometry = gpd.read_parquet(geometry_cache)
    print(f'Loaded geometry cache: {geometry_cache}')
else:
    feats = fetch_arcgis_records(PARCEL_QUERY_URL, out_fields=['OBJECTID', 'PIN'], return_geometry=True)
    rows = []
    for f in feats:
        rings = f['geometry'].get('rings', [])
        geom = Polygon(rings[0], holes=rings[1:]) if rings else None
        rows.append({'OBJECTID': f['attributes']['OBJECTID'], 'PIN': f['attributes']['PIN'], 'geometry': geom})
    parcel_geometry = (
        gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
        .drop_duplicates('OBJECTID').sort_values('OBJECTID').reset_index(drop=True)
    )
    parcel_geometry.to_parquet(DATA_DIR / 'baltimore_geometry_fresh.parquet')

gdf = parcel_geometry.merge(parcel_attrs, on=['OBJECTID', 'PIN'], how='inner')
gdf = gdf.drop_duplicates(subset=['OBJECTID']).sort_values('OBJECTID').reset_index(drop=True)
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')
print(f'Citywide parcels: {len(gdf):,}')

## Step 2b: Assessment groups

`ASSESGRP` comes from a separate live pull (the layer caps at 1,000 records per request). Each PIN maps to exactly one group, so duplicate-PIN accounts inherit the group cleanly.

In [ ]:
ASSESGRP_PATH = _latest('baltimore_assesgrp_*.parquet')
if ASSESGRP_PATH is not None:
    assesgrp = pd.read_parquet(ASSESGRP_PATH)
    print(f'Loaded assessment-group cache: {ASSESGRP_PATH}')
else:
    feats = fetch_arcgis_records(PARCEL_QUERY_URL, out_fields=['OBJECTID', 'PIN', 'ASSESGRP'])
    assesgrp = (
        pd.DataFrame([f['attributes'] for f in feats])
        .drop_duplicates('OBJECTID').reset_index(drop=True)
    )
    assesgrp.to_parquet(DATA_DIR / 'baltimore_assesgrp_fresh.parquet', index=False)

assert assesgrp.groupby('PIN')['ASSESGRP'].nunique().max() == 1, 'A PIN spans multiple assessment groups'
pin_to_group = assesgrp.drop_duplicates('PIN').set_index('PIN')['ASSESGRP'].astype(str).str.strip()

gdf['ASSESGRP'] = gdf['PIN'].map(pin_to_group)
unmatched = gdf['ASSESGRP'].isna() | ~gdf['ASSESGRP'].isin(GROUPS)
print(f'Parcels without a valid assessment group: {int(unmatched.sum()):,} of {len(gdf):,} — excluded from modeling')
gdf = gdf[~unmatched].copy()
print(gdf['ASSESGRP'].value_counts().sort_index().to_string())

In [ ]:
# Clean values and flag full exemptions (same logic as the citywide model)
numeric_cols = [
    'CURRLAND', 'CURRIMPR', 'EXMPLAND', 'EXMPIMPR', 'BFCVLAND', 'BFCVIMPR',
    'ARTAXBAS', 'CITY_TAX', 'CITYCRED', 'CCREDAMT', 'STATCRED', 'SCREDAMT',
    'DWELUNIT', 'YEAR_BUILD', 'STRUCTAREA',
]
for col in numeric_cols:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0)

text_cols = ['ZONECODE', 'USEGROUP', 'DHCDUSE1', 'NO_IMPRV', 'VACIND', 'OWNER_1', 'OWNER_2', 'FULLADDR', 'NEIGHBOR', 'PROPDESC']
for col in text_cols:
    if col in gdf.columns:
        gdf[col] = gdf[col].fillna('').astype(str).str.strip()

gdf['full_exmp'] = ((gdf['ARTAXBAS'] <= 0) | (gdf['CITY_TAX'] <= 0)).astype(int)

# Land/improvement basis: allocate the taxed base (ARTAXBAS) by current full cash shares.
# BFCVLAND is NULL for ~46% of parcels (verified against SDAT records 2026-06-11), so the
# BFCV base-cycle components cannot carry the land signal; CURRLAND/CURRIMPR match SDAT's
# current "Value" column exactly.
curr_total = gdf['CURRLAND'] + gdf['CURRIMPR']
land_share = (gdf['CURRLAND'] / curr_total).where(curr_total > 0)
gdf['taxable_land_value'] = (gdf['ARTAXBAS'].clip(lower=0) * land_share).fillna(0.0)
gdf['taxable_improvement_value'] = (gdf['ARTAXBAS'].clip(lower=0) * (1 - land_share)).fillna(0.0)
gdf['hold_out'] = (gdf['ARTAXBAS'] > 0) & (gdf['CITY_TAX'] > 0) & (curr_total <= 0)
print(f"Fully exempt (no current city bill): {int(gdf['full_exmp'].sum()):,} of {len(gdf):,}")
print(f"Billed parcels without a current land/improvement split (held at current tax): {int(gdf['hold_out'].sum()):,}")

# Per-account lot size parsed from the assessor's LOT_SIZE string ('14X83-10',
# '1.953 ACRES', '741 SF IMP. ONLY'). Parcel-polygon area is unreliable here:
# improvement-only accounts share the whole community's polygon (e.g. 1101 Horners Lane),
# so geometry-based areas overstate those lots by orders of magnitude.
import re

def parse_lot_size(s):
    if not isinstance(s, str):
        return np.nan
    s = s.strip().upper().replace(',', '')
    try:
        m = re.match(r'^(\d+(?:\.\d+)?)\s*ACRE', s)
        if m:
            return float(m.group(1)) * 43560.0
        m = re.match(r'^(\d+(?:\.\d+)?)\s*S\.?\s*Q?\.?\s*F', s)
        if m:
            return float(m.group(1))
        m = re.match(r'^(\d+)(?:-(\d+))?(?:\.\d+)?\s*X\s*(\d+)(?:-(\d+))?', s)
        if m:
            front = float(m.group(1)) + (float(m.group(2)) / 12.0 if m.group(2) else 0.0)
            depth = float(m.group(3)) + (float(m.group(4)) / 12.0 if m.group(4) else 0.0)
            return front * depth
    except (ValueError, TypeError):
        return np.nan
    return np.nan

gdf['lot_sqft'] = gdf['LOT_SIZE'].map(parse_lot_size)
print(f"Lot size parsed: {gdf['lot_sqft'].notna().mean()*100:.2f}% of parcels")

## Step 3: Property categories

Same Baltimore-specific hybrid mapping as the citywide and transit-zone models.

In [ ]:
# Fully exempt parcels (no current city bill) are excluded from the modeled universe,
# exports, and report graphics. They pay $0 under both systems and only dilute the
# category and quintile statistics (e.g. ~61% of Vacant Land is city-owned exempt).
gdf = gdf[gdf['full_exmp'] == 0].copy()
print(f"Modeling universe: {len(gdf):,} billed parcels (fully exempt removed)")


def categorize_baltimore_property(row):
    zone_code = str(row.get('ZONECODE', '')).strip().upper()
    usegroup = str(row.get('USEGROUP', '')).strip().upper()
    dwell_units = pd.to_numeric(row.get('DWELUNIT', 0), errors='coerce')
    curr_impr = pd.to_numeric(row.get('CURRIMPR', 0), errors='coerce')
    lot_sqft = pd.to_numeric(row.get('lot_sqft'), errors='coerce')
    no_imprv = str(row.get('NO_IMPRV', '')).strip().upper()

    if (curr_impr <= 0) or (no_imprv == 'Y'):
        return 'Vacant Land'

    if usegroup == 'R':
        if dwell_units <= 1:
            # 1/4 acre (10,890 sq ft) isolates the top ~4% of SFH lots (median lot is ~1,650 sq ft)
            if lot_sqft >= 10_890:
                return 'Single Family Residential (1/4+ acre)'
            return 'Single Family Residential'
        if 2 <= dwell_units <= 4:
            return 'Small Multi-Family (2-4 units)'
        if dwell_units >= 5:
            return 'Large Multi-Family (5+ units)'
        return 'Other Residential'

    if zone_code.startswith('R-'):
        return 'Other Residential'

    if zone_code.startswith('I-'):
        return 'Industrial'

    if zone_code.startswith('OR-') or zone_code.startswith('C-') or zone_code.startswith('EC-') or zone_code.startswith('CC') or zone_code.startswith('PC-') or zone_code.startswith('TOD-') or zone_code in {'BSC', 'IMU-1', 'MI', 'H', 'OS'}:
        return 'Commercial / Mixed Use'

    if usegroup in {'C', 'CC', 'CR', 'RC', 'EC'}:
        return 'Commercial / Mixed Use'

    if usegroup == 'I':
        return 'Industrial'

    if usegroup == 'U':
        return 'Utility / Special'

    if usegroup == 'E':
        return 'Institutional / Exempt'

    if usegroup == 'M':
        return 'Large Multi-Family (5+ units)'

    return 'Other'


gdf['PROPERTY_CATEGORY'] = gdf.apply(categorize_baltimore_property, axis=1)
display(
    pd.crosstab(gdf['PROPERTY_CATEGORY'], gdf['ASSESGRP'])
)

## Step 4: Current tax per group

The citywide derived millage (`CITY_TAX / ARTAXBAS`) anchors the validation; observed `CITY_TAX` is `current_tax`. Each group's revenue-neutral target is its own observed levy.

In [ ]:
city_millage = round((gdf['CITY_TAX'].sum() / gdf['ARTAXBAS'].sum()) * 1000.0, 4)
print(f'Derived citywide millage: {city_millage:.4f} mills')

group_rows = []
group_data = {}
for g in GROUPS:
    zone = gdf[gdf['ASSESGRP'] == g].copy()
    zone['city_millage'] = city_millage
    target = float(zone['CITY_TAX'].sum())

    modeled_revenue, _, zone = calculate_current_tax(
        df=zone,
        tax_value_col='ARTAXBAS',
        millage_rate_col='city_millage',
        exemption_flag_col='full_exmp',
    )
    zone['modeled_current_tax'] = zone['current_tax']
    zone['current_tax'] = zone['CITY_TAX']

    gap_pct = (modeled_revenue / target - 1) * 100
    assert abs(gap_pct) < 5.0, f'Group {g} revenue gap {gap_pct:.1f}% exceeds threshold'
    group_data[g] = {'df': zone, 'target': target}
    group_rows.append({
        'group': g, 'parcels': len(zone), 'exempt': int(zone['full_exmp'].sum()),
        'observed_levy': target, 'modeled_from_artaxbas': modeled_revenue, 'gap_pct': gap_pct,
    })

display(pd.DataFrame(group_rows).round({'gap_pct': 2}))

## Step 5: 4:1 split-rate, one independent solve per group

In [ ]:
summary_rows = []
for g in GROUPS:
    zone = group_data[g]['df']
    target = group_data[g]['target']
    land_millage, improvement_millage, new_revenue, zone = model_split_rate_tax(
        df=zone,
        land_value_col='taxable_land_value',
        improvement_value_col='taxable_improvement_value',
        current_revenue=target,
        land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
        exemption_flag_col='full_exmp',
        exclude_mask=zone['hold_out'],
    )
    group_data[g].update({'df': zone, 'land_millage': land_millage, 'improvement_millage': improvement_millage})

    land_share = zone['taxable_land_value'].sum() / (zone['taxable_land_value'].sum() + zone['taxable_improvement_value'].sum())
    summary_rows.append({
        'group': g, 'target_revenue': target, 'new_revenue': new_revenue,
        'land_millage': land_millage, 'improvement_millage': improvement_millage,
        'taxable_land_share': land_share,
    })

display(pd.DataFrame(summary_rows).round({'land_millage': 4, 'improvement_millage': 4, 'taxable_land_share': 4}))

In [ ]:
for g in GROUPS:
    zone = group_data[g]['df']
    category_summary = calculate_category_tax_summary(
        df=zone,
        category_col='PROPERTY_CATEGORY',
        current_tax_col='current_tax',
        new_tax_col='new_tax',
        pct_threshold=10.0,
    )
    print_category_tax_summary(
        summary_df=category_summary,
        title=f'4:1 Split-Rate Within SDAT Assessment Group {g} - Baltimore, MD',
        pct_threshold=10.0,
    )

# Cross-group comparison: median % change by category
pivot = pd.concat(
    [group_data[g]['df'].assign(group=g) for g in GROUPS], ignore_index=True
).groupby(['PROPERTY_CATEGORY', 'group'])['tax_change_pct'].median().unstack('group').round(1)
print('Median tax change % by category and assessment group:')
display(pivot)

## Step 6: Exploration charts

In [ ]:
MD_STATE_PLANE = 'EPSG:26985'
fig, ax = plt.subplots(figsize=(9, 10))
colors = {'1': '#1b9e77', '2': '#d95f02', '3': '#7570b3'}
gdf_proj = gdf.to_crs(MD_STATE_PLANE)
pts = gdf_proj.geometry.centroid
for g in GROUPS:
    m = gdf_proj['ASSESGRP'] == g
    ax.scatter(pts[m].x, pts[m].y, c=colors[g], s=0.5, alpha=0.5, label=f'Group {g}')
ax.legend(markerscale=20)
ax.set_axis_off()
ax.set_title('Baltimore SDAT assessment groups (parcel centroids)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'groups_map.png', dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(12, 7))
pivot.plot(kind='barh', ax=ax, color=[colors[g] for g in pivot.columns])
ax.axvline(0, color='black', linewidth=1, linestyle='dotted')
ax.set_xlabel('Median % change')
ax.set_title('Median tax change % by category and assessment group (4:1 split-rate)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'group_category_preview.png', dpi=150)
plt.close()
print('Saved exploration charts to data/')

## Step 7: Census join and standard exports

One county fetch (Baltimore city FIPS 24510), then each group is spatially matched, exported, and reported as its own city.

In [ ]:
# Census fetch — once for all three groups
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
census_gdf = None
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
        except concurrent.futures.TimeoutError:
            print('Census API timed out — exports will carry NaN census columns')
except Exception as e:
    print(f'Census fetch failed: {e} — exports will carry NaN census columns')

In [ ]:
from lvt.viz import create_city_report

for g in GROUPS:
    zone = group_data[g]['df']
    city_name = f'{CITY_BASE}{g}'
    if census_gdf is not None:
        zone = match_to_census_blockgroups(zone, census_gdf)
        # census_gdf already carries demographics — spatial join adds them above.
        # Do NOT do a second merge with census_data: it creates median_income_x/y
        # duplicates and silently zeros out all demographic output.
        if 'minority_pct' not in zone.columns and 'total_pop' in zone.columns and 'white_pop' in zone.columns:
            zone['minority_pct'] = ((zone['total_pop'] - zone['white_pop']) / zone['total_pop'] * 100).round(2)
        if 'black_pct' not in zone.columns and 'total_pop' in zone.columns and 'black_pop' in zone.columns:
            zone['black_pct'] = (zone['black_pop'] / zone['total_pop'] * 100).round(2)
        print(f"Group {g} census join: {zone['std_geoid'].notna().mean()*100:.1f}% matched")
    else:
        for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
            zone[_col] = float('nan')

    out_df = save_standard_export(
        df=zone,
        city=city_name,
        output_path=f'../../analysis/data/{city_name}.csv',
        model_type=MODEL_TYPE,
        land_millage=group_data[g]['land_millage'],
        improvement_millage=group_data[g]['improvement_millage'],
        property_category_col='PROPERTY_CATEGORY',
        current_tax_col='current_tax',
        new_tax_col='new_tax',
        tax_change_col='tax_change',
        tax_change_pct_col='tax_change_pct',
        taxable_land_col='taxable_land_value',
        taxable_improvement_col='taxable_improvement_value',
        exempt_flag_col='full_exmp',
    )
    create_city_report(out_df, city_name, show=False)

print('Done.')

## Validation Summary (re-based 2026-06-11)

| Check | Group 1 | Group 2 | Group 3 |
|---|---|---|---|
| Parcels (billed, modeled) | 68,935 | 81,683 | 69,523 |
| Parcels incl. fully exempt (excluded) | 72,354 | 90,848 | 75,263 |
| Observed levy (target) | $382.4M | $373.3M | $401.5M |
| Revenue match (ARTAXBAS × 22.2914 mills) | −0.82% | −0.82% | −0.83% |
| Revenue neutrality | exact | exact | exact |
| Land / improvement millage | 53.38 / 13.35 | 53.79 / 13.45 | 51.46 / 12.87 |
| Taxable land share | 22.8% | 22.4% | 24.9% |
| Census coverage | 99.8% | 100.0% | 99.9% |
| Report PNGs | 7 | 7 | 7 |
| SFH (<1/4 acre) median change | −0.2% | +7.7% | −5.2% |
| SFH (1/4+ acre) median change | +6.4% | +1.2% | −4.3% |

Basis and interpretation:

- **Land/improvement basis (portal-verified 2026-06-11):** the taxed base `ARTAXBAS` is allocated by current full cash shares (`CURRLAND`/`CURRIMPR`). The earlier run used the `BFCV*` base-cycle fields, whose land component is NULL at group-correlated rates (56%/41%/41% of parcels) — that artifact manufactured the previously reported SFR divergence of −16.1% / +55.8% / +31.7%. On the corrected basis the groups look nearly alike (land shares 22.8% / 22.4% / 24.9%, near-identical millages), and the remaining SFR spread (+0.4% / +7.6% / −5.2%) reflects genuine composition differences.
- Each group's `CURR*` values date to that group's most recent triennial reassessment, so a mild vintage asymmetry remains across groups (Group 1 was revalued 1/1/2026; Groups 2–3 carry older current values).
- Billed parcels without a current land/improvement split (606 citywide) are held at current tax via `exclude_mask`; parcels with no current city bill are flagged fully exempt.